In [1]:
# ============================
# 📦 Essential Libraries
# ============================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# ============================================
# 📂 Load all raw CSV datasets (collaborative-safe)
# ============================================

import os
import pandas as pd

# 1) Locate project root by searching for "data" folder
BASE_DIR = os.getcwd()
while "data" not in os.listdir(BASE_DIR) and os.path.dirname(BASE_DIR) != BASE_DIR:
    BASE_DIR = os.path.dirname(BASE_DIR)

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")

print("Project root:", BASE_DIR)
print("Raw data folder:", RAW_DIR)
print("Files in raw:", os.listdir(RAW_DIR))
print("=====================================\n")

# 2) Load each dataset into a named DataFrame

df_cpi = pd.read_csv(os.path.join(RAW_DIR, "ons_cpi.csv"), low_memory=False)
df_interest = pd.read_csv(os.path.join(RAW_DIR, "boe_interest.csv"))
df_exchange = pd.read_csv(os.path.join(RAW_DIR, "exchange_rates.csv"))
df_gdp = pd.read_csv(os.path.join(RAW_DIR, "GDP_growth_Rate.csv"))
df_unemp = pd.read_csv(os.path.join(RAW_DIR, "Unemployment_Rate.csv"))
df_oil = pd.read_csv(os.path.join(RAW_DIR, "petrol_oil_everage_price_change.csv"))

print("Loaded DataFrames:")
print("  df_cpi      -> ons_cpi.csv")
print("  df_interest -> boe_interest.csv")
print("  df_exchange -> exchange_rates.csv")
print("  df_gdp      -> GDP_growth_Rate.csv")
print("  df_unemp    -> Unemployment_Rate.csv")
print("  df_oil      -> petrol_oil_everage_price_change.csv")


Project root: c:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI
Raw data folder: c:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI\data\raw
Files in raw: ['.Rhistory', 'boe_interest.csv', 'exchange_rates.csv', 'GDP_growth_Rate.csv', 'ons_cpi.csv', 'petrol_oil_everage_price_change.csv', 'Unemployment_Rate.csv']

Loaded DataFrames:
  df_cpi      -> ons_cpi.csv
  df_interest -> boe_interest.csv
  df_exchange -> exchange_rates.csv
  df_gdp      -> GDP_growth_Rate.csv
  df_unemp    -> Unemployment_Rate.csv
  df_oil      -> petrol_oil_everage_price_change.csv


## 🎯 CPI Column Selection  
This step extracts the two key variables required from the CPI dataset:

- **Title** — the date or period identifier  
- **CPI ANNUAL RATE 00: ALL ITEMS 2015=100** — the annual inflation rate (main target variable)

A separate working DataFrame (`cpi_clean`) is created to allow further cleaning and transformation without modifying the original dataset.


In [3]:
# Extract two important CPI columns
cpi_clean = df_cpi[["Title", "CPI ANNUAL RATE 00: ALL ITEMS 2015=100"]].copy()
cpi_clean.head(10000)


,Title,CPI ANNUAL RATE 00: ALL ITEMS 2015=100
0,CDID,D7G7
1,PreUnit,NaN
2,Unit,%
3,Release Date,22-10-2025
4,Next release,19 November 2025
...,...,...
1476,2025 MAY,3.4
1477,2025 JUN,3.6
1478,2025 JUL,3.8
1479,2025 AUG,3.8


In [4]:
cpi_clean.dtypes


Title                                     object
CPI ANNUAL RATE 00: ALL ITEMS 2015=100    object
dtype: object

## 🧹 CPI Data Cleaning  
This step performs the initial cleaning of the CPI dataset:

- Converts `Title` to a proper datetime format  
- Converts the inflation variable to numeric values  
- Renames `Title` to `Date` to standardise the time index  

This prepares the CPI data for later merging with other economic indicators.


In [5]:
# Month patterns
months = "JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC"

# Keep only rows that contain a month abbreviation
cpi_monthly = cpi_clean[cpi_clean["Title"].str.contains(months, na=False)].copy()

# Drop fully empty rows
cpi_monthly = cpi_monthly.dropna(how="all")

# Drop rows where Title or CPI value is missing
cpi_monthly = cpi_monthly.dropna(subset=["Title", "CPI ANNUAL RATE 00: ALL ITEMS 2015=100"])

# Show first few cleaned rows
cpi_monthly.head(10)


,Title,CPI ANNUAL RATE 00: ALL ITEMS 2015=100
1040,1989 JAN,4.9
1041,1989 FEB,5.0
1042,1989 MAR,5.0
1043,1989 APR,5.3
1044,1989 MAY,5.3
1045,1989 JUN,5.2
1046,1989 JUL,5.2
1047,1989 AUG,5.0
1048,1989 SEP,5.2
1049,1989 OCT,5.5


In [6]:
cpi_monthly["Date"] = pd.to_datetime(cpi_monthly["Title"], format="%Y %b", errors="coerce")


In [7]:
cpi_monthly["CPI ANNUAL RATE 00: ALL ITEMS 2015=100"] = pd.to_numeric(
    cpi_monthly["CPI ANNUAL RATE 00: ALL ITEMS 2015=100"],
    errors="coerce"
)


In [8]:
cpi_monthly.head(100)

,Title,CPI ANNUAL RATE 00: ALL ITEMS 2015=100,Date
1040,1989 JAN,4.9,1989-01-01
1041,1989 FEB,5.0,1989-02-01
1042,1989 MAR,5.0,1989-03-01
1043,1989 APR,5.3,1989-04-01
1044,1989 MAY,5.3,1989-05-01
...,...,...,...
1135,1996 DEC,2.3,1996-12-01
1136,1997 JAN,2.1,1997-01-01
1137,1997 FEB,1.9,1997-02-01
1138,1997 MAR,1.7,1997-03-01


## 🧹 Removing Redundant Columns  
After converting the `Title` field into a proper datetime variable (`Date`),  
the original `Title` column becomes unnecessary.  
This step removes it, leaving only the cleaned `Date` column and the CPI annual inflation rate.


In [9]:
cpi_monthly = cpi_monthly.drop(columns=["Title"])
cpi_monthly.head()


,CPI ANNUAL RATE 00: ALL ITEMS 2015=100,Date
1040,4.9,1989-01-01
1041,5.0,1989-02-01
1042,5.0,1989-03-01
1043,5.3,1989-04-01
1044,5.3,1989-05-01


## 🎯 GDP Column Extraction  
This step selects the date column (`Title`) and the key GDP indicator  
**“Gross Value Added – Monthly (3 month on 3 month growth) : CVM SA”**.  
The resulting DataFrame (`gdp_clean`) will be cleaned and prepared for merging with other monthly datasets.


In [10]:
gdp_clean = df_gdp[[
    "Title",
    "Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA"
]].copy()


In [11]:
gdp_clean.dtypes
gdp_clean.head(100)



,Title,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA
0,CDID,ED3H
1,PreUnit,NaN
2,Unit,NaN
3,Release Date,16-10-2025
4,Next release,13 November 2025
...,...,...
95,2004 JUN,0.4
96,2004 JUL,0.5
97,2004 AUG,0.4
98,2004 SEP,0.3


## 🧹 GDP 3-Month Growth Cleaning  
This step filters the GDP dataset to monthly observations, converts the `Title` field into a proper datetime variable (`Date`),  
and converts the selected GDP growth series  
**"Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA"**  
to a numeric variable named `GDP_3m3m_Growth`.  
The cleaned data is sorted by date and stored in `gdp_monthly` for later merging.


In [12]:
# Keep only rows that contain a month abbreviation in Title
months = "JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC"

gdp_monthly = gdp_clean[gdp_clean["Title"].str.contains(months, na=False)].copy()

# Drop fully empty rows
gdp_monthly = gdp_monthly.dropna(how="all")

# Drop rows where Title or GDP value is missing
gdp_col = "Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA"
gdp_monthly = gdp_monthly.dropna(subset=["Title", gdp_col])

gdp_monthly.head(10)


,Title,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA
11,1997 JUN,1.0
12,1997 JUL,0.6
13,1997 AUG,0.9
14,1997 SEP,0.8
15,1997 OCT,1.0
16,1997 NOV,1.0
17,1997 DEC,1.4
18,1998 JAN,1.4
19,1998 FEB,1.4
20,1998 MAR,0.8


In [13]:
# Convert Title to datetime (same pattern as CPI, e.g. "1989 JAN")
gdp_monthly["Date"] = pd.to_datetime(gdp_monthly["Title"], format="%Y %b", errors="coerce")

# Convert GDP growth column to numeric
gdp_monthly[gdp_col] = pd.to_numeric(gdp_monthly[gdp_col], errors="coerce")

# Drop old Title column
gdp_monthly = gdp_monthly.drop(columns=["Title"])


# Sort by date and reset index
gdp_monthly = gdp_monthly.sort_values("Date").reset_index(drop=True)

gdp_monthly.head()


,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA,Date
0,1.0,1997-06-01
1,0.6,1997-07-01
2,0.9,1997-08-01
3,0.8,1997-09-01
4,1.0,1997-10-01


## 🏦 Interest Rate Dataset Overview  
This step provides an initial inspection of the interest rate dataset.  
The output includes:

- Total number of rows and columns  
- First ten observations for visual inspection  
- A complete list of column names  

This overview is used to identify the date field and the primary Bank Rate series before cleaning and selection.


In [14]:
print("Shape:", df_interest.shape)
print("=====================================\n")

print("First 10 rows:")
display(df_interest.head(1000))
print("=====================================\n")

print("Column names:")
for col in df_interest.columns:
    print("-", col)


Shape: (2526, 2)

First 10 rows:


,Date,Bank Rate
0,27-11-2015,0.50
1,30-11-2015,0.50
2,01-12-2015,0.50
3,02-12-2015,0.50
4,03-12-2015,0.50
...,...,...
995,05-11-2019,0.75
996,06-11-2019,0.75
997,07-11-2019,0.75
998,08-11-2019,0.75



Column names:
- Date
- Bank Rate


## 🏦 Interest Rate Monthly Aggregation  
The daily Bank Rate series is converted into a monthly series by:

1. Parsing the `Date` column as a proper datetime (DD–MM–YYYY).  
2. Ensuring `Bank Rate` is stored as a numeric variable.  
3. Creating a `YearMonth` key and, after sorting by date,  
   selecting the last observation in each month to represent the monthly Bank Rate.  

The resulting cleaned series is stored in `interest_monthly`.


In [15]:
# ============================================
# 🏦 Clean interest rate data to monthly series
# ============================================

# 1) Convert Date to datetime (day-first format: DD-MM-YYYY)
df_interest["Date"] = pd.to_datetime(df_interest["Date"], dayfirst=True, errors="coerce")

# 2) Ensure Bank Rate is numeric
df_interest["Bank Rate"] = pd.to_numeric(df_interest["Bank Rate"], errors="coerce")

# 3) Create Year-Month key
df_interest["YearMonth"] = df_interest["Date"].dt.to_period("M")

# 4) Sort by date so "last in month" really means latest date
df_interest = df_interest.sort_values("Date")

# 5) Keep the latest observation in each month
interest_monthly = df_interest.groupby("YearMonth").tail(1).copy()

# 6) Drop helper column and tidy up
interest_monthly = interest_monthly.drop(columns=["YearMonth"])
interest_monthly = interest_monthly.sort_values("Date").reset_index(drop=True)

# 7) Quick check
interest_monthly.head(1000)


,Date,Bank Rate
0,2015-11-30,0.50
1,2015-12-31,0.50
2,2016-01-29,0.50
3,2016-02-29,0.50
4,2016-03-31,0.50
...,...,...
116,2025-07-31,4.25
117,2025-08-29,4.00
118,2025-09-30,4.00
119,2025-10-31,4.00


## 💱 Exchange Rate Monthly Cleaning  
This step prepares the monthly exchange-rate series by:

1. Keeping only monthly rows using month-name filtering.  
2. Converting the `Title` field to a proper datetime (`Date`).  
3. Converting the USD exchange rate to numeric format.  
4. Selecting the latest observation in each month to form a monthly series.  

The final cleaned dataset is stored as `exchange_monthly` with a standardised column name `Exchange_USD`.



In [16]:
df_exchange.columns


Index(['Title', 'Average Sterling exchange rate: US Dollar XUMAUSS'], dtype='object')

In [17]:
months = "JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC"

exchange_clean = df_exchange[df_exchange["Title"].str.contains(months, na=False)].copy()

# Drop fully empty rows
exchange_clean = exchange_clean.dropna(how="all")

# Drop rows missing Title or Rate
rate_col = "Average Sterling exchange rate: US Dollar XUMAUSS"
exchange_clean = exchange_clean.dropna(subset=["Title", rate_col])


In [18]:
# Convert Title → datetime
exchange_clean["Date"] = pd.to_datetime(exchange_clean["Title"], format="%Y %b", errors="coerce")

# Convert rate → numeric
exchange_clean[rate_col] = pd.to_numeric(exchange_clean[rate_col], errors="coerce")

# Drop old Title column
exchange_clean = exchange_clean.drop(columns=["Title"])


In [19]:
exchange_clean["YearMonth"] = exchange_clean["Date"].dt.to_period("M")

# Sort by date so the last row of each month is the latest
exchange_clean = exchange_clean.sort_values("Date")

# Keep latest row in each month
exchange_monthly = exchange_clean.groupby("YearMonth").tail(1).copy()

# Final cleaning
exchange_monthly = exchange_monthly.drop(columns=["YearMonth"])
exchange_monthly = exchange_monthly.sort_values("Date").reset_index(drop=True)

# Rename column
exchange_monthly = exchange_monthly.rename(columns={
    rate_col: "Exchange_USD"
})

exchange_monthly.head(12)


,Exchange_USD,Date
0,1.6587,1997-01-01
1,1.6246,1997-02-01
2,1.6063,1997-03-01
3,1.6295,1997-04-01
4,1.6334,1997-05-01
5,1.6446,1997-06-01
6,1.6702,1997-07-01
7,1.6034,1997-08-01
8,1.6015,1997-09-01
9,1.6329,1997-10-01


## 📉 Unemployment Rate Monthly Cleaning  
This step converts the unemployment dataset to a clean monthly series by:

1. Filtering rows that contain a valid month name.  
2. Parsing the `Title` field into a datetime `Date` column.  
3. Converting the unemployment rate to numeric format.  
4. Selecting the latest available value in each month.  

The final cleaned dataset is stored in `unemployment_monthly` with a standardised name `Unemployment_Rate`.


In [20]:
df_unemp.columns


Index(['Title', 'Unemployment rate (aged 16 and over, seasonally adjusted): %'], dtype='object')

In [21]:
months = "JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC"

unemp_clean = df_unemp[df_unemp["Title"].str.contains(months, na=False)].copy()

# Drop blank rows
unemp_clean = unemp_clean.dropna(how="all")

# Drop rows missing date or value
unemp_col = "Unemployment rate (aged 16 and over, seasonally adjusted): %"
unemp_clean = unemp_clean.dropna(subset=["Title", unemp_col])


In [22]:
# Convert Title → datetime
unemp_clean["Date"] = pd.to_datetime(unemp_clean["Title"], format="%Y %b", errors="coerce")

# Convert unemployment value → numeric
unemp_clean[unemp_col] = pd.to_numeric(unemp_clean[unemp_col], errors="coerce")

# Drop the old Title column
unemp_clean = unemp_clean.drop(columns=["Title"])


In [23]:
unemp_clean.head(100)

,"Unemployment rate (aged 16 and over, seasonally adjusted): %",Date
279,3.8,1971-02-01
280,3.9,1971-03-01
281,4.0,1971-04-01
282,4.1,1971-05-01
283,4.1,1971-06-01
...,...,...
374,5.3,1979-01-01
375,5.4,1979-02-01
376,5.3,1979-03-01
377,5.3,1979-04-01


## ⛽ Oil Price Dataset Overview  
This step reports the size of the oil dataset, displays the first ten observations,  
and lists all column names to identify the date field and the relevant oil price series  
for subsequent cleaning.


In [24]:
# ============================================
# ⛽ Oil Data Overview (petrol_oil_everage_price_change.csv)
# ============================================

print("Shape:", df_oil.shape)
print("=====================================\n")

print("First 10 rows:")
display(df_oil.head(10))
print("=====================================\n")

print("Column names:")
for col in df_oil.columns:
    print("-", col)


Shape: (649, 2)

First 10 rows:


,Title,RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil
0,CDID,DOGQ
1,Source dataset ID,MM23
2,PreUnit,NaN
3,Unit,%
4,Release date,19-11-2025
5,Next release,17 December 2025
6,Important notes,NaN
7,1988,-1.7
8,1989,7.2
9,1990,12.7



Column names:
- Title
- RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil


In [25]:
months = "JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC"

oil_clean = df_oil[df_oil["Title"].str.contains(months, na=False)].copy()

# Drop fully empty rows
oil_clean = oil_clean.dropna(how="all")

oil_col = "RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil"

# Drop rows missing date or oil value
oil_clean = oil_clean.dropna(subset=["Title", oil_col])


In [26]:
# Convert Title → datetime
oil_clean["Date"] = pd.to_datetime(oil_clean["Title"], format="%Y %b", errors="coerce")

# Convert oil price % change → numeric
oil_clean[oil_col] = pd.to_numeric(oil_clean[oil_col], errors="coerce")

# Drop old Title column
oil_clean = oil_clean.drop(columns=["Title"])


In [27]:
oil_clean.head(100)

,RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil,Date
195,-1.3,1988-01-01
196,-4.1,1988-02-01
197,-3.8,1988-03-01
198,-1.9,1988-04-01
199,-1.7,1988-05-01
...,...,...
290,6.2,1995-12-01
291,5.4,1996-01-01
292,3.5,1996-02-01
293,2.1,1996-03-01


## 🗓️ Step 1 — Create a Monthly Key (YearMonth)

Different datasets contain dates in different formats.  
Some have monthly values recorded on the first day of the month (e.g., `2021-01-01`),  
while others contain multiple daily observations per month, from which the latest value  
was selected (e.g., `2021-01-31`).  
Although all represent monthly data, the date values do not align perfectly,  
which would cause mismatches during merging.

To standardise the merge across all datasets, a common **YearMonth** key is created  
from the existing `Date` column using the `.dt.to_period("M")` method.  
This converts each date into a monthly period such as `2021-01`, ensuring  
all datasets can be merged cleanly at the monthly frequency.

A `YearMonth` column is added to every cleaned dataset for use as the consistent merge key.


In [28]:
for df in [cpi_monthly, gdp_monthly, interest_monthly, exchange_monthly, unemp_clean, oil_clean]:
    df["YearMonth"] = df["Date"].dt.to_period("M")


In [29]:
# ============================================
# 🔍 View all datasets after adding YearMonth
# ============================================

datasets = {
    "CPI Monthly": cpi_monthly,
    "GDP Monthly": gdp_monthly,
    "Interest Monthly": interest_monthly,
    "Exchange Monthly": exchange_monthly,
    "Unemployment Monthly": unemp_clean,
    "Oil Monthly": oil_clean
}

for name, df in datasets.items():
    print(f"\n===== {name} =====")
    print(df.head(10))
    print(df.info())
    print("\n")



===== CPI Monthly =====
      CPI ANNUAL RATE 00: ALL ITEMS 2015=100       Date YearMonth
1040                                     4.9 1989-01-01   1989-01
1041                                     5.0 1989-02-01   1989-02
1042                                     5.0 1989-03-01   1989-03
1043                                     5.3 1989-04-01   1989-04
1044                                     5.3 1989-05-01   1989-05
1045                                     5.2 1989-06-01   1989-06
1046                                     5.2 1989-07-01   1989-07
1047                                     5.0 1989-08-01   1989-08
1048                                     5.2 1989-09-01   1989-09
1049                                     5.5 1989-10-01   1989-10
<class 'pandas.core.frame.DataFrame'>
Index: 441 entries, 1040 to 1480
Data columns (total 3 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----    

## 🧩 Purpose of the Final Merge Code

The final merge step combines all cleaned monthly datasets  
(CPI, GDP, interest rate, exchange rate, unemployment rate, and oil price)  
into one unified table indexed by month.  
The key points of this process are:

### 1. Removing Duplicate Date Columns
Only the CPI dataset retains its `Date` column.  
Other datasets have their `Date` column removed because all datasets will be aligned  
using the `YearMonth` key.  
This prevents duplicated `Date_x`, `Date_y` fields after merging.  
No data rows are deleted in this step—only extra date columns are removed.

### 2. Merging on `YearMonth`
All datasets are merged using an **outer join** on the `YearMonth` key.  
This ensures that:
- All CPI months remain intact.
- Each variable appears from its natural start date.
- No observations are dropped, even if some variables start later than others.
- Months with missing indicators simply contain `NaN` for those fields,  
  which is standard in macroeconomic time-series.

### 3. Reconstructing a Monthly Date
After merging, a consistent monthly `Date` column is created from the `YearMonth`  
key using `to_timestamp(how="start")`.  
This assigns each row the first day of the month (e.g., `1997-06-01`),  
ensuring a uniform date format for analysis, plotting, and modeling.

### ✔ Outcome
The result is a complete monthly dataset that preserves the full historical range  
of every variable and aligns all indicators correctly by month.  
This merged dataset becomes the foundation for feature engineering,  
model training (XGBoost, LSTM, hybrid), and explainability (SHAP, LIME).


In [30]:
for df in [gdp_monthly, interest_monthly, exchange_monthly, unemp_clean, oil_clean]:
    df.drop(columns=["Date"], inplace=True)


In [31]:
merged = cpi_monthly.copy()

merged = merged.merge(gdp_monthly,   on="YearMonth", how="outer")
merged = merged.merge(interest_monthly, on="YearMonth", how="outer")
merged = merged.merge(exchange_monthly, on="YearMonth", how="outer")
merged = merged.merge(unemp_clean,   on="YearMonth", how="outer")
merged = merged.merge(oil_clean,     on="YearMonth", how="outer")

merged = merged.sort_values("YearMonth").reset_index(drop=True)


In [32]:
merged["Date"] = merged["YearMonth"].dt.to_timestamp(how="start")


In [34]:
merged.head(10000)

,CPI ANNUAL RATE 00: ALL ITEMS 2015=100,Date,YearMonth,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA,Bank Rate,Exchange_USD,"Unemployment rate (aged 16 and over, seasonally adjusted): %",RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil
0,NaN,1971-02-01,1971-02,NaN,NaN,NaN,3.8,NaN
1,NaN,1971-03-01,1971-03,NaN,NaN,NaN,3.9,NaN
2,NaN,1971-04-01,1971-04,NaN,NaN,NaN,4.0,NaN
3,NaN,1971-05-01,1971-05,NaN,NaN,NaN,4.1,NaN
4,NaN,1971-06-01,1971-06,NaN,NaN,NaN,4.1,NaN
...,...,...,...,...,...,...,...,...
653,3.8,2025-07-01,2025-07,0.2,4.25,1.3492,4.8,-7.0
654,3.8,2025-08-01,2025-08,0.3,4.00,1.3450,NaN,-5.2
655,3.8,2025-09-01,2025-09,NaN,4.00,NaN,NaN,-1.4
656,NaN,2025-10-01,2025-10,NaN,4.00,NaN,NaN,1.1


## 💾 Saving the Final Merged Dataset  
The unified monthly dataset is exported as a CSV file to the  
`data/processed` directory.  
This file will be used for feature engineering, model training,  
and all subsequent analysis steps.


In [35]:
# ============================================
# 💾 Save merged dataset in the current folder
# ============================================

merged.to_csv("merged_monthly_dataset.csv", index=False)

print("File saved as: merged_monthly_dataset.csv")
print("Location:", os.getcwd())


File saved as: merged_monthly_dataset.csv
Location: c:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI\data\processed
